# 01 — Silver SIAF
## Transformación Bronze → Silver

Este notebook procesa los datos crudos del SIAF (Sistema Integrado de Administración Financiera) desde la Capa Bronze hacia la Capa Silver.

Fuente: `data/bronze/siaf/{año}/batch_*.json`  
Destino: `data/silver/siaf/` (Parquet particionado por ANO_DOC)

**Transformaciones aplicadas:**
- Limpieza de falsos nulos (GHOST_NULLS)
- Normalización de strings y casteos de tipos
- Filtro: solo municipalidades (NIVEL_GOBIERNO == 'M')
- Deduplicación por llave de negocio
- Variables analíticas derivadas: UBIGEO, PERFORMANCE_RATE, BUDGET_GAP
- Detección de anomalías en catálogos de referencia
- 12 DQ Flags de calidad de datos

In [1]:
import sys, os, time, shutil
from pathlib import Path

# Agregar raíz del proyecto al path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, ByteType
)
from pyspark.sql.window import Window

from src.paths import BRONZE, SILVER, str_path, ensure_dirs
from src.spark_utils import get_spark, write_parquet
from src.transformations.common import GHOST_NULLS, clean_ghost_nulls, print_dq_summary
from core.audit.control_manager import ControlManager, ExecutionStatus

print("Imports OK")

Imports OK


In [2]:
# ── Configuración del Pipeline ────────────────────────────────────────────────
YEARS = [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]          # Años a procesar
BRONZE_PATH = str_path(BRONZE["siaf"])
SILVER_PATH = str_path(SILVER["siaf"])

print(f"Bronze source : {BRONZE_PATH}")
print(f"Silver target : {SILVER_PATH}")
print(f"Años a procesar: {YEARS}")

ensure_dirs()

Bronze source : /workspace/data/bronze/siaf
Silver target : /workspace/data/silver/siaf
Años a procesar: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
[OK] Estructura de directorios verificada en /workspace/data


In [3]:
# ── SparkSession ──────────────────────────────────────────────────────────────
spark = get_spark(app_name="MEF_Silver_SIAF", memory="4g")

[OK] SparkSession lista | version=3.5.0 | master=local[*]


## PARTE 1: Lectura de la Capa Bronze

El schema se define explícitamente para evitar inferencia incorrecta de tipos.
Todos los campos llegan como string desde el JSON y se castean en el paso siguiente.

In [4]:
# ── Schema explícito del JSON Bronze ──────────────────────────────────────────
BRONZE_SCHEMA = StructType([
    StructField("ANO_DOC",                       StringType()),
    StructField("MES_DOC",                       StringType()),
    StructField("NIVEL_GOBIERNO",                StringType()),
    StructField("NIVEL_GOBIERNO_NOMBRE",         StringType()),
    StructField("SECTOR",                        StringType()),
    StructField("SECTOR_NOMBRE",                 StringType()),
    StructField("PLIEGO",                        StringType()),
    StructField("PLIEGO_NOMBRE",                 StringType()),
    StructField("EJECUTORA",                     StringType()),
    StructField("EJECUTORA_NOMBRE",              StringType()),
    StructField("SEC_EJEC",                      StringType()),
    StructField("DEPARTAMENTO_EJECUTORA",        StringType()),
    StructField("DEPARTAMENTO_EJECUTORA_NOMBRE", StringType()),
    StructField("PROVINCIA_EJECUTORA",           StringType()),
    StructField("PROVINCIA_EJECUTORA_NOMBRE",    StringType()),
    StructField("DISTRITO_EJECUTORA",            StringType()),
    StructField("DISTRITO_EJECUTORA_NOMBRE",     StringType()),
    StructField("FUENTE_FINANCIAMIENTO",         StringType()),
    StructField("FUENTE_FINANCIAMIENTO_NOMBRE",  StringType()),
    StructField("RUBRO",                         StringType()),
    StructField("RUBRO_NOMBRE",                  StringType()),
    StructField("TIPO_RECURSO",                  StringType()),
    StructField("TIPO_RECURSO_NOMBRE",           StringType()),
    StructField("GENERICA",                      StringType()),
    StructField("GENERICA_NOMBRE",               StringType()),
    StructField("SUBGENERICA",                   StringType()),
    StructField("SUBGENERICA_NOMBRE",            StringType()),
    StructField("SUBGENERICA_DET",               StringType()),
    StructField("SUBGENERICA_DET_NOMBRE",        StringType()),
    StructField("ESPECIFICA",                    StringType()),
    StructField("ESPECIFICA_NOMBRE",             StringType()),
    StructField("ESPECIFICA_DET",                StringType()),
    StructField("ESPECIFICA_DET_NOMBRE",         StringType()),
    StructField("MONTO_PIA",                     StringType()),
    StructField("MONTO_PIM",                     StringType()),
    StructField("MONTO_RECAUDADO",               StringType()),
])

# Construir lista de paths válidos por año
bronze_paths = []
for year in YEARS:
    y_path = BRONZE["siaf"] / str(year)
    if y_path.exists():
        bronze_paths.append(str_path(y_path))
        print(f"  Año {year}: {y_path}")
    else:
        print(f"   Año {year}: sin datos en {y_path}")

if not bronze_paths:
    raise FileNotFoundError(f"No hay datos Bronze en {BRONZE_PATH}. Ejecuta primero: python -m bronze.ingest_bronze")

# Lectura unificada de todos los años
raw_df = spark.read.schema(BRONZE_SCHEMA).option("multiline", "false").json(bronze_paths)
n_raw = raw_df.count()
print(f"\nRegistros Bronze cargados: {n_raw:,}")
raw_df.printSchema()

  Año 2012: /workspace/data/bronze/siaf/2012
  Año 2013: /workspace/data/bronze/siaf/2013
  Año 2014: /workspace/data/bronze/siaf/2014
  Año 2015: /workspace/data/bronze/siaf/2015
  Año 2016: /workspace/data/bronze/siaf/2016
  Año 2017: /workspace/data/bronze/siaf/2017
  Año 2018: /workspace/data/bronze/siaf/2018
  Año 2019: /workspace/data/bronze/siaf/2019
  Año 2020: /workspace/data/bronze/siaf/2020
  Año 2021: /workspace/data/bronze/siaf/2021
  Año 2022: /workspace/data/bronze/siaf/2022
  Año 2023: /workspace/data/bronze/siaf/2023
  Año 2024: /workspace/data/bronze/siaf/2024
  Año 2025: /workspace/data/bronze/siaf/2025
  Año 2026: /workspace/data/bronze/siaf/2026

Registros Bronze cargados: 10,915,638
root
 |-- ANO_DOC: string (nullable = true)
 |-- MES_DOC: string (nullable = true)
 |-- NIVEL_GOBIERNO: string (nullable = true)
 |-- NIVEL_GOBIERNO_NOMBRE: string (nullable = true)
 |-- SECTOR: string (nullable = true)
 |-- SECTOR_NOMBRE: string (nullable = true)
 |-- PLIEGO: string (

## PARTE 2: Limpieza y Normalización

### Falsos Nulos (Ghost Nulls)
Los JSONs de la API contienen valores como `""`, `"None"`, `"N/A"`, etc.
que deben convertirse a `NULL` real antes de cualquier transformación.

In [5]:
# ── 1. Falsos Nulos ───────────────────────────────────────────────────────────
df = raw_df.cache()  # Cachear para evitar re-lecturas costosas
df = clean_ghost_nulls(df)
print(f"Ghost nulls limpiados en todas las columnas string")

Ghost nulls limpiados en todas las columnas string


In [6]:
# ── 2. Casteos de tipos ───────────────────────────────────────────────────────
FINANCIAL_METRICS = ["MONTO_PIA", "MONTO_PIM", "MONTO_RECAUDADO"]

for c in FINANCIAL_METRICS:
    df = df.withColumn(c, F.col(c).cast(DoubleType()))
df = df.withColumn("ANO_DOC", F.col("ANO_DOC").cast(IntegerType()))
df = df.withColumn("MES_DOC", F.col("MES_DOC").cast(IntegerType()))

# ── 3. Filtros de negocio ─────────────────────────────────────────────────────
# Mantener solo registros de municipalidades con al menos un monto
df = df.filter(
    (F.col("MONTO_PIA").isNotNull() | F.col("MONTO_PIM").isNotNull() | F.col("MONTO_RECAUDADO").isNotNull()) &
    (F.col("NIVEL_GOBIERNO") == "M")
)
n_filtered = df.count()
print(f"Registros después de filtro (solo municipalidades con montos): {n_filtered:,}")
print(f"   Eliminados: {n_raw - n_filtered:,} ({(n_raw - n_filtered)/n_raw*100:.1f}%)")

Registros después de filtro (solo municipalidades con montos): 8,999,260
   Eliminados: 1,916,378 (17.6%)


In [7]:
total = df.count()

null_pliego = df.filter(F.col("PLIEGO").isNull()).count()

# 3. Registros con PLIEGO NO null (por si acaso hay ruido)
not_null_pliego = df.filter(F.col("PLIEGO").isNotNull()).count()

# 4. Distribución de PLIEGO si existe data
print("Resumen general:")
print(f"  Total registros        : {total:,}")
print(f"  PLIEGO NULL            : {null_pliego:,} ({null_pliego/total*100:.2f}%)")
print(f"  PLIEGO NO NULL         : {not_null_pliego:,} ({not_null_pliego/total*100:.2f}%)")

# 5. Top valores no nulos (si existen)
if not_null_pliego > 0:
    print("\n Top PLIEGOS encontrados:")
    (
        df.filter(F.col("PLIEGO").isNotNull())
          .groupBy("PLIEGO")
          .count()
          .orderBy("count", ascending=False)
          .show(20, truncate=False)
    )
else:
    print("\n No hay PLIEGO con valores en municipalidades (100% NULL o vacío)")

# 6. Validación explícita
if null_pliego == total:
    print("\n CONFIRMACIÓN: PLIEGO es 100% NULL en municipalidades")
    print("Puede eliminarse del modelo sin pérdida de información")
else:
    print("\n ALERTA: PLIEGO tiene valores en municipalidades")
    print("Revisar consistencia del filtro o del origen de datos")

print("\n================ FIN AUDITORÍA PLIEGO ================\n")

Resumen general:
  Total registros        : 8,999,260
  PLIEGO NULL            : 8,999,260 (100.00%)
  PLIEGO NO NULL         : 0 (0.00%)

 No hay PLIEGO con valores en municipalidades (100% NULL o vacío)

 CONFIRMACIÓN: PLIEGO es 100% NULL en municipalidades
Puede eliminarse del modelo sin pérdida de información

================ FIN AUDITORÍA PLIEGO ================



In [8]:
# ── Limpieza estructural ───────────────

if "PLIEGO" in df.columns:
    df = df.drop("PLIEGO", "PLIEGO_NOMBRE")

In [9]:
# ── 4. Fill NAs en métricas financieras ──────────────────────────────────────
for c in FINANCIAL_METRICS:
    df = df.fillna({c: 0.0})

# ── 5. Deduplicación por llave de negocio ─────────────────────────────────────
CORE_IDENTITY = [
    "ANO_DOC", "MES_DOC", "SEC_EJEC", "EJECUTORA",
    "FUENTE_FINANCIAMIENTO", "RUBRO", "TIPO_RECURSO",
    "GENERICA", "SUBGENERICA", "SUBGENERICA_DET", "ESPECIFICA", "ESPECIFICA_DET",
]
w_dedup = Window.partitionBy(CORE_IDENTITY).orderBy(F.col("MONTO_PIM").desc())
df = df.withColumn("_rn", F.row_number().over(w_dedup)).filter(F.col("_rn") == 1).drop("_rn")
n_dedup = df.count()
print(f"Registros tras deduplicación: {n_dedup:,} (eliminados {n_filtered - n_dedup:,} duplicados)")

# ── 6. Fill NAs geográficos (Mancomunidades sin ubigeo asignado) ──────────────
for c in ["DEPARTAMENTO_EJECUTORA", "PROVINCIA_EJECUTORA", "DISTRITO_EJECUTORA"]:
    df = df.fillna({c: "00"})
for c in ["DEPARTAMENTO_EJECUTORA_NOMBRE", "PROVINCIA_EJECUTORA_NOMBRE", "DISTRITO_EJECUTORA_NOMBRE"]:
    df = df.fillna({c: "SIN ASIGNAR"})
print("Fill NAs geográficos aplicado")

Registros tras deduplicación: 8,999,223 (eliminados 37 duplicados)
Fill NAs geográficos aplicado


## PARTE 3: Variables Analíticas Derivadas

| Campo | Descripción |
|-------|-------------|
| `UBIGEO` | Código de 6 dígitos (Dept+Prov+Dist) |
| `PERFORMANCE_RATE` | MONTO_RECAUDADO / MONTO_PIM |
| `BUDGET_GAP` | MONTO_PIM - MONTO_RECAUDADO |
| `PERFORMANCE_STATUS` | LOW / MEDIUM / HIGH según tasa de avance |
| `PERIOD_DATE` | Fecha de corte como tipo Date (yyyy-MM-01) |

In [10]:
# ── 7. Variables analíticas ───────────────────────────────────────────────────
df = (
    df
    .withColumn("UBIGEO", F.concat(
        F.lpad(F.col("DEPARTAMENTO_EJECUTORA"), 2, "0"),
        F.lpad(F.col("PROVINCIA_EJECUTORA"), 2, "0"),
        F.lpad(F.col("DISTRITO_EJECUTORA"), 2, "0")
    ))
    .withColumn("PERFORMANCE_RATE",
        F.when(F.col("MONTO_PIM") > 0, F.col("MONTO_RECAUDADO") / F.col("MONTO_PIM"))
         .otherwise(F.lit(None).cast(DoubleType()))
    )
    .withColumn("BUDGET_GAP", F.col("MONTO_PIM") - F.col("MONTO_RECAUDADO"))
    .withColumn("PERFORMANCE_STATUS",
        F.when(F.col("PERFORMANCE_RATE").isNull(), "UNKNOWN")
         .when(F.col("PERFORMANCE_RATE") <= 0.6, "LOW")
         .when(F.col("PERFORMANCE_RATE") <= 0.8, "MEDIUM")
         .otherwise("HIGH")
    )
    .withColumn("PERIOD_DATE",
        F.to_date(
            F.concat(F.col("ANO_DOC").cast("string"), F.lit("-"),
                     F.lpad(F.col("MES_DOC").cast("string"), 2, "0"), F.lit("-01")),
            "yyyy-MM-dd"
        )
    )
)
print("Variables analíticas derivadas creadas")
df.select("UBIGEO", "PERFORMANCE_RATE", "BUDGET_GAP", "PERFORMANCE_STATUS", "PERIOD_DATE").show(5, truncate=False)

Variables analíticas derivadas creadas
+------+--------------------+----------+------------------+-----------+
|UBIGEO|PERFORMANCE_RATE    |BUDGET_GAP|PERFORMANCE_STATUS|PERIOD_DATE|
+------+--------------------+----------+------------------+-----------+
|010101|0.0                 |24000.0   |LOW               |2012-01-01 |
|010101|NULL                |0.0       |UNKNOWN           |2012-01-01 |
|010101|0.047880000000000006|952.12    |LOW               |2012-01-01 |
|010101|0.0                 |100.0     |LOW               |2012-01-01 |
|010101|0.073283            |9267.17   |LOW               |2012-01-01 |
+------+--------------------+----------+------------------+-----------+
only showing top 5 rows



## PARTE 4: Data Quality Flags

12 flags binarios (0/1) que clasifican la calidad de cada registro:

| Flag | Condición |
|------|----------|
| DQ_NEG_PIA | PIA < 0 |
| DQ_PIM_LT_PIA | PIM < PIA |
| DQ_NEG_PIM | PIM < 0 |
| DQ_NEG_REC | RECAUDADO < 0 |
| DQ_HIGH_RATIO | RECAUDADO > 2×PIM |
| DQ_ABSURD_AMOUNT | Monto > 50,000 millones |
| DQ_INCOMPLETE_RECORD | Campos clave nulos |
| DQ_INVALID_GOV_LEVEL | Nivel gobierno no E/R/M |
| DQ_INVALID_FUNDING | Fuente financiamiento fuera de [1,9] |
| DQ_INVALID_GEOGRAPHY | UBIGEO no es 6 dígitos |
| DQ_BROKEN_HIERARCHY | EJECUTORA sin PLIEGO |
| DQ_CATALOG_MISMATCH | Código con múltiples descripciones |

In [11]:
# ── 8. Detección de anomalías de catálogo ─────────────────────────────────────
HIERARCHY_MAPPINGS = [
    ("NIVEL_GOBIERNO",         "NIVEL_GOBIERNO_NOMBRE"),
    ("FUENTE_FINANCIAMIENTO",  "FUENTE_FINANCIAMIENTO_NOMBRE"),
    ("RUBRO",                  "RUBRO_NOMBRE"),
    ("GENERICA",               "GENERICA_NOMBRE"),
    ("DEPARTAMENTO_EJECUTORA", "DEPARTAMENTO_EJECUTORA_NOMBRE"),
    ("SECTOR",                 "SECTOR_NOMBRE"),
]

catalog_anomalies = {}
catalog_df = df.select(*{c for pair in HIERARCHY_MAPPINGS for c in pair}).distinct()

for code_col, name_col in HIERARCHY_MAPPINGS:
    agg_df = (
        catalog_df.filter(F.col(code_col).isNotNull())
        .groupBy(code_col).agg(F.countDistinct(name_col).alias("n"))
        .filter(F.col("n") > 1)
    )
    bad_codes = [row[code_col] for row in agg_df.collect()]
    if bad_codes:
        catalog_anomalies[code_col] = bad_codes
        print(f"   {code_col}: {len(bad_codes)} código(s) con múltiples descripciones")

if not catalog_anomalies:
    print("Catálogos de referencia sin anomalías")

Catálogos de referencia sin anomalías


In [12]:
# ── 9. Aplicar DQ Flags ───────────────────────────────────────────────────────
ABSURD_THRESHOLD = 50_000_000_000.0

cond_incompleto = (
    F.col("ANO_DOC").isNull() | F.col("MES_DOC").isNull() |
    F.col("NIVEL_GOBIERNO").isNull() | F.col("EJECUTORA").isNull() |
    F.col("FUENTE_FINANCIAMIENTO").isNull() | F.col("RUBRO").isNull() |
    F.col("GENERICA").isNull() | F.col("ESPECIFICA_DET").isNull()
)

mismatch_cond = F.lit(False)
for col_cod, bad_codes in catalog_anomalies.items():
    if bad_codes:
        mismatch_cond = mismatch_cond | F.col(col_cod).isin(bad_codes)

df = (
    df
    .withColumn("DQ_NEG_PIA",           F.when(F.col("MONTO_PIA") < 0, 1).otherwise(0).cast(ByteType()))
    .withColumn("DQ_PIM_LT_PIA",        F.when(F.col("MONTO_PIM") < F.col("MONTO_PIA"), 1).otherwise(0).cast(ByteType()))
    .withColumn("DQ_NEG_PIM",           F.when(F.col("MONTO_PIM") < 0, 1).otherwise(0).cast(ByteType()))
    .withColumn("DQ_NEG_REC",           F.when(F.col("MONTO_RECAUDADO") < 0, 1).otherwise(0).cast(ByteType()))
    .withColumn("DQ_HIGH_RATIO",        F.when((F.col("MONTO_PIM") > 0) & (F.col("MONTO_RECAUDADO") > F.col("MONTO_PIM") * 2), 1).otherwise(0).cast(ByteType()))
    .withColumn("DQ_ABSURD_AMOUNT",     F.when((F.abs(F.col("MONTO_PIA")) > ABSURD_THRESHOLD) | (F.abs(F.col("MONTO_PIM")) > ABSURD_THRESHOLD) | (F.abs(F.col("MONTO_RECAUDADO")) > ABSURD_THRESHOLD), 1).otherwise(0).cast(ByteType()))
    .withColumn("DQ_INCOMPLETE_RECORD", F.when(cond_incompleto, 1).otherwise(0).cast(ByteType()))
    .withColumn("DQ_INVALID_GOV_LEVEL", F.when(~F.col("NIVEL_GOBIERNO").isin("E", "R", "M"), 1).otherwise(0).cast(ByteType()))
    .withColumn("DQ_INVALID_FUNDING",   F.when(~F.col("FUENTE_FINANCIAMIENTO").cast("int").between(1, 9), 1).otherwise(0).cast(ByteType()))
    .withColumn("DQ_INVALID_GEOGRAPHY", F.when(~F.col("UBIGEO").rlike(r"^\d{6}$"), 1).otherwise(0).cast(ByteType()))
    .withColumn("DQ_BROKEN_HIERARCHY",  F.when(F.col("EJECUTORA").isNotNull() & F.col("SEC_EJEC").isNull(), 1).otherwise(0).cast(ByteType()))
    .withColumn("DQ_CATALOG_MISMATCH",  F.when(mismatch_cond, 1).otherwise(0).cast(ByteType()))
)

DQ_COLS = [
    "DQ_NEG_PIA", "DQ_PIM_LT_PIA", "DQ_NEG_PIM", "DQ_NEG_REC",
    "DQ_HIGH_RATIO", "DQ_ABSURD_AMOUNT", "DQ_INCOMPLETE_RECORD",
    "DQ_INVALID_GOV_LEVEL", "DQ_INVALID_FUNDING", "DQ_INVALID_GEOGRAPHY",
    "DQ_BROKEN_HIERARCHY", "DQ_CATALOG_MISMATCH",
]
print(f"{len(DQ_COLS)} DQ Flags aplicados")

12 DQ Flags aplicados


In [13]:
# ── 10. Resumen de calidad ────────────────────────────────────────────────────
print_dq_summary(df, DQ_COLS)


📊 Resumen de Quality Flags:
  Flag                             Registros con flag
  ----------------------------------------------------
  DQ_NEG_PIA                              0  (0.0%)
  DQ_PIM_LT_PIA                      31,824  (0.4%)
  DQ_NEG_PIM                         13,548  (0.2%)
  DQ_NEG_REC                         63,426  (0.7%)
  DQ_HIGH_RATIO                       7,863  (0.1%)
  DQ_ABSURD_AMOUNT                        0  (0.0%)
  DQ_INCOMPLETE_RECORD                    0  (0.0%)
  DQ_INVALID_GOV_LEVEL                    0  (0.0%)
  DQ_INVALID_FUNDING                      0  (0.0%)
  DQ_INVALID_GEOGRAPHY                    0  (0.0%)
  DQ_BROKEN_HIERARCHY                     0  (0.0%)
  DQ_CATALOG_MISMATCH                     0  (0.0%)


## PARTE 5: Escritura a Silver (Parquet particionado por ANO_DOC)

Usamos `partitionBy("ANO_DOC")` para que Spark organice los archivos por año,
permitiendo lectura eficiente por filtro de año en las etapas Gold.

In [14]:
# ── 11. Escritura Parquet ─────────────────────────────────────────────────────
control = ControlManager(pipeline_name="silver_siaf")
execution = control.start(input_parameters={"years": YEARS})

start_time = time.time()
try:
    # Borrado seguro de particiones existentes para reescritura limpia
    for year in YEARS:
        part_path = SILVER["siaf"] / f"ANO_DOC={year}"
        if part_path.exists():
            shutil.rmtree(str(part_path))
            print(f"Partición existente eliminada: ANO_DOC={year}")

    n_final = write_parquet(df, SILVER_PATH, mode="append", partition_by=["ANO_DOC"])

    elapsed = time.time() - start_time
    control.end(
        status=ExecutionStatus.SUCCESS,
        output_summary={"raw_rows": n_raw, "clean_rows": n_final, "elapsed_sec": round(elapsed, 1)}
    )
    print(f"\nSilver SIAF completado en {elapsed:.1f}s")
    print(f"   {n_raw:,} raw → {n_final:,} Silver")
except Exception as e:
    control.log_error("SilverSIAFError", str(e))
    control.end(status=ExecutionStatus.FAILED, output_summary={"error": str(e)})
    raise

2026-06-19 18:21:28 | INFO     | mef_dw.audit.control_manager | [AUDIT] Ejecución iniciada | pipeline=silver_siaf id=b4ff0cf2
  ✅ 8,999,223 filas → /workspace/data/silver/siaf
2026-06-19 18:22:12 | INFO     | mef_dw.audit.control_manager | [AUDIT] Ejecución terminada | id=b4ff0cf2 status=SUCCESS duration=43.7s
2026-06-19 18:22:12 | INFO     | mef_dw.audit.control_manager | [AUDIT] Reporte guardado: /workspace/data/audit/executions/20260619_182212_b4ff0cf2_silver_siaf.json

Silver SIAF completado en 43.7s
   10,915,638 raw → 8,999,223 Silver


In [15]:
# ── 12. Verificación ──────────────────────────────────────────────────────────
print("\nVerificación de la salida Silver:")
df_check = spark.read.parquet(SILVER_PATH)
print(f"  Total filas: {df_check.count():,}")
print(f"  Columnas: {len(df_check.columns)}")
print("  Particiones (ANO_DOC):")
df_check.groupBy("ANO_DOC").count().orderBy("ANO_DOC").show()
df_check.printSchema()


Verificación de la salida Silver:
  Total filas: 8,999,223
  Columnas: 51
  Particiones (ANO_DOC):
+-------+------+
|ANO_DOC| count|
+-------+------+
|   2012|638925|
|   2013|590349|
|   2014|611717|
|   2015|632600|
|   2016|613538|
|   2017|550948|
|   2018|527741|
|   2019|571750|
|   2020|572621|
|   2021|691765|
|   2022|638548|
|   2023|630891|
|   2024|643442|
|   2025|739985|
|   2026|344403|
+-------+------+

root
 |-- MES_DOC: integer (nullable = true)
 |-- NIVEL_GOBIERNO: string (nullable = true)
 |-- NIVEL_GOBIERNO_NOMBRE: string (nullable = true)
 |-- SECTOR: string (nullable = true)
 |-- SECTOR_NOMBRE: string (nullable = true)
 |-- EJECUTORA: string (nullable = true)
 |-- EJECUTORA_NOMBRE: string (nullable = true)
 |-- SEC_EJEC: string (nullable = true)
 |-- DEPARTAMENTO_EJECUTORA: string (nullable = true)
 |-- DEPARTAMENTO_EJECUTORA_NOMBRE: string (nullable = true)
 |-- PROVINCIA_EJECUTORA: string (nullable = true)
 |-- PROVINCIA_EJECUTORA_NOMBRE: string (nullable = tr